# MNE test

In [1]:
import mne
from pathlib import Path
from scipy import signal

# Loading data

In [ ]:
file = '/data/BioPAP/data/CBN4IL230113.bdf'
raw = mne.io.read_raw_bdf(file, preload=True)

In [ ]:
file = '/data/BioPAP/data/cropped.fif'
# file = '/data/BioPAP/data/altered.fif'
# raw = mne.io.read_raw_fif(file)
raw = mne.io.Raw(file, preload=True, verbose=True)

In [ ]:
raw.info

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

raw.plot_psd(tmin=500, picks='eeg')
plt.show()

# 0. Anonymization

In [5]:
raw.info.keys()

# raw.info['subject_info'].keys()

# The only entries that should be manually changed by the user are
# info['bads'], info['description'], info['device_info'], info['dev_head_t'], info['experimenter'], 
# info['helium_info'], info['line_freq'], info['temp'] and info['subject_info'].

dict_keys(['file_id', 'events', 'hpi_results', 'hpi_meas', 'subject_info', 'device_info', 'helium_info', 'hpi_subsystem', 'proc_history', 'meas_id', 'experimenter', 'description', 'proj_id', 'proj_name', 'meas_date', 'utc_offset', 'sfreq', 'highpass', 'lowpass', 'line_freq', 'gantry_angle', 'chs', 'dev_head_t', 'ctf_head_t', 'dev_ctf_t', 'dig', 'bads', 'ch_names', 'nchan', 'projs', 'comps', 'acq_pars', 'acq_stim', 'custom_ref_applied', 'xplotter_layout', 'kit_system_id'])

In [6]:
from datetime import datetime
from datetime import timezone

raw.set_meas_date(datetime.now(tz=timezone.utc))
raw.info['experimenter'] = 'A. Scientist'
raw.info['subject_info'] = {'his_id': '-'}

#TODO: what about filename? randomize?
#TODO: description

In [ ]:
raw.anonymize(verbose=True)

# mne.io.anonymize_info(raw.info, daysback=100)
# raw.anonymize(daysback=100, verbose=True)

In [ ]:
# raw.info
for key in raw.info.keys():
    print(f"{key:<15} {raw.info[key]}")

In [ ]:
# https://mne.tools/stable/generated/mne.io.Raw.html#mne.io.Raw.plot
raw.plot(n_channels=1)

In [ ]:
raw.crop(tmax=2000)

In [ ]:
raw.save('/data/BioPAP/data/altered.fif')

# 1. Rereference to mastoid

In [10]:
# https://mne.tools/stable/generated/mne.set_eeg_reference.html#mne.set_eeg_reference
# raw: Array of reference data subtracted from EEG channels.
# ref_data: Array of reference data subtracted from EEG channels.

# 'Channels to which the new reference applies to: ': cannot set explicitly (but it's all of them, right?)

ref_channels=['EXG1', 'EXG2']
raw, ref_data = mne.set_eeg_reference(raw, ref_channels=ref_channels, copy=False)

EEG channel type selected for re-referencing
Applying a custom ('EEG',) reference.


In [ ]:
raw.ch_names

# 2. Create hEOG and vEOG

In [33]:
ref_channels=['hEOG', 'vEOG']

exist_channels = [x for x in ref_channels if x in raw.ch_names]
raw.drop_channels(exist_channels)

# raw = mne.add_reference_channels(raw, ref_channels=ref_channels, copy=False)
mne.set_bipolar_reference(raw, anode=['EXG3','EXG5'], cathode=['EXG4','EXG6'], ch_name=['hEOG','vEOG'], copy=False, drop_refs=False) 

# A bipolar reference takes the difference between two channels (the anode minus the cathode) and adds it as a new virtual channel.
# "The original channels will be dropped." ???

# print(raw.ch_names)

raw[['hEOG', 'vEOG']]

EEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=2, n_times=2048001
    Range : 0 ... 2048000 =      0.000 ...  1000.000 secs
Ready.
Added the following bipolar channels:
hEOG, vEOG


(array([[-0.00690533, -0.00690052, -0.00689764, ..., -0.00198506,
         -0.00198425, -0.00198821],
        [-0.01541882, -0.01541772, -0.01541547, ..., -0.01606016,
         -0.01605775, -0.01605131]], shape=(2, 2048001)),
 array([0.00000000e+00, 4.88281250e-04, 9.76562500e-04, ...,
        9.99999023e+02, 9.99999512e+02, 1.00000000e+03], shape=(2048001,)))

# 3. Filtering

## Option 1

In [34]:
# Zero phase shift Butterworth filters.
# Global filter settings:
# Low cutoff: 0.1 Hz, time constant 1.5915494309189535, order 4
# High cutoff: 35 Hz, order 4 
# Notch filter: ---

# https://mne.tools/stable/generated/mne.filter.construct_iir_filter.html#mne.filter.construct_iir_filter
l_freq=0.1
h_freq=35

iir_params=dict(order=4, ftype='butter')

filtered = mne.filter.filter_data(
    data=raw.get_data(),
    sfreq=raw.info['sfreq'],
    l_freq=l_freq,
    h_freq=h_freq,
    method='iir',
    iir_params=iir_params,
    verbose=True,
    copy=False)

Setting up band-pass filter from 0.1 - 35 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 0.10, 35.00 Hz: -6.02, -6.02 dB



## Option 2

In [37]:
filtered = raw.filter(l_freq=l_freq,
           h_freq=h_freq,
           method='iir',
           iir_params=iir_params, 
           verbose=True)

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.1 - 35 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 0.10, 35.00 Hz: -6.02, -6.02 dB



# 4. Downsampling

In [39]:
# Downsampling
# *** Change Sampling Rate / Resolution ***
# Conversion is based on spline interpolation.
# New Sampling Rate [Hz]: 500
# New Sampling Interval [µS]: 2000
# Data was filtered before downsampling with 225Hz, 24dB/oct.

# interval?
# downsampling?

sfreq = 500

raw.notch_filter(freqs=225)

events = mne.find_events(raw)
raw, events = raw.resample(sfreq=sfreq, events=events, verbose=True)

Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 2.2e+02 - 2.3e+02 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 223.94
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 223.69 Hz)
- Upper passband edge: 226.06 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 226.31 Hz)
- Filter length: 3301 samples (6.602 s)

Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
1168 events found on stim channel Status
Event IDs: [  1   2   3   5   9  10  11  12  13  14  15  16  17  18  19  20  21  22
  23  24  25  26  27  28  29  30  31  32  33  34  35  36  37  38  39  50
  51  52  53  54  55  56 

# 5. Macro to check for missing markers

# 6. Macro to copy trial and block markers at the start of the trial

# 7. Segmentation Critical words S1-6

In [ ]:
# *** Segmentation ***
# Segmentation relative to reference marker positions
# Reference markers:
#     Stimulus    S  6
#     Stimulus    S  5
#     Stimulus    S  4
#     Stimulus    S  3
#     Stimulus    S  2
#     Stimulus    S  1
# Advanced Boolean Expression:    ---
# Segment size and position relative to reference markers:
# Start: -200.00 ms, End: 1000.00 ms, Length: 1200.00 ms
# Allow overlapped segments? No
# Skip bad intervals?        No
# Data was cached to a temporary file.

# 99. Gabriel

In [ ]:
raw.set_channel_types({'hEOG':'eog','vEOG':'eog','EXG4':'misc','EXG5':'misc','Erg1':'misc',
                      'EXG1':'misc','EXG2':'misc','EXG3':'misc','EXG4':'misc'})#declare type to avoid confusion with EEG channels
#Renaming electrodes